# EdgeNetV4-B2 — Ablation Study Harness
**7 cumulative configs × 50 epochs each · Seed 42 · Auto-skip checkpointing**

**Run cells in order. Cell 8 is the only long-running cell (~3.5–4.5 hrs total).**

| Cell | What it does | Run once? |
|------|-------------|----------|
| 1 | Config, imports, seed, constants | Once |
| 2 | Scan raw datasets → DataFrame (no CSV needed) | Once |
| 3 | Splits + Dataset class + DataLoaders | Once |
| 4 | Model architecture (flags: b2, CA, ms_head) | Once |
| 5 | Losses, EMA, CutMix, metrics functions | Once |
| 6 | `train_one_config()` trainer | Once |
| 7 | 7 ablation configs list | Once |
| 8 | **RUN LOOP** (change `RUN_IDX` to run one at a time) | 7× or 'all' |
| 9 | Results table + staircase plot + per-class F1 heatmap | After runs |

In [11]:
# ════════════════════════════════════════════════════════════════════════
# CELL 1 — CONFIG + IMPORTS + SEED
# ════════════════════════════════════════════════════════════════════════
import os, gc, json, time, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

import torchvision.transforms as T
from PIL import Image
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

warnings.filterwarnings('ignore')

# ── DATASET PATHS (kagglehub, do NOT move these) ──────────────────────────
MVTEC_PATH    = Path('/home/sufi/.cache/kagglehub/datasets/hycloud/mvtech-anomaly-detection/versions/1/mvtech_anomaly_detection')
CASTING_PATH  = Path('/home/sufi/.cache/kagglehub/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product/versions/2')
MAGNETIC_PATH = Path('/home/sufi/.cache/kagglehub/datasets/alex000kim/magnetic-tile-surface-defects/versions/1')

# ── OUTPUT (ablation results land here) ──────────────────────────────────
OUTPUT_DIR  = Path('/home/sufi/training_results/ablation')
RESULTS_DIR = OUTPUT_DIR / 'run_results'    # one JSON per completed run
CKPT_DIR    = OUTPUT_DIR / 'checkpoints'    # best weights per run
for d in [OUTPUT_DIR, RESULTS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── PROTOCOL (LOCKED — do not change) ────────────────────────────────────
SEED       = 42
EPOCHS     = 50
BATCH_SIZE = 32
NUM_WORKERS = 4
IMG_SIZE   = 224
GRAD_CLIP  = 1.0

# ── 8 SEMANTIC DEFECT GROUPS (ablation uses grouped, not 54 fine-grained) ─
SEMANTIC_GROUPS   = ['contamination', 'cut', 'deformation', 'fracture',
                     'hole_void', 'minor_defect', 'scratch', 'surface_quality']
NUM_DEFECT_TYPES  = len(SEMANTIC_GROUPS)  # 8
NUM_PRODUCT_TYPES = 17
DEFECT_TO_IDX     = {n: i for i, n in enumerate(SEMANTIC_GROUPS)}

# ── DEFECT FOLDER NAME → SEMANTIC GROUP ──────────────────────────────────
DEFECT_GROUP_MAP = {
    # fracture
    'crack':'fracture','fracture':'fracture','faulty_imprint':'fracture',
    'broken':'fracture','broken_large':'fracture','broken_small':'fracture',
    'MT_Break':'fracture','MT_Crack':'fracture',
    'broken_teeth':'fracture','split_teeth':'fracture','damaged_case':'fracture',
    # hole_void
    'hole':'hole_void','void':'hole_void','pit':'hole_void',
    'blowhole':'hole_void','MT_Blowhole':'hole_void',
    # scratch
    'scratch':'scratch','score':'scratch','MT_Fray':'scratch',
    'scratch_head':'scratch','scratch_neck':'scratch',
    # deformation
    'bent':'deformation','bent_lead':'deformation','squeeze':'deformation',
    'squeezed_teeth':'deformation','bent_wire':'deformation',
    'fold':'deformation','flip':'deformation',
    # contamination
    'contamination':'contamination','glue':'contamination','oil':'contamination',
    'glue_strip':'contamination','liquid':'contamination',
    'metal_contamination':'contamination','gray_stroke':'contamination',
    # surface_quality
    'stain':'surface_quality','color':'surface_quality','rough':'surface_quality',
    'uneven':'surface_quality','inclusion':'surface_quality',
    'discolor':'surface_quality','pilling':'surface_quality',
    'MT_Uneven':'surface_quality','def_front':'surface_quality','print':'surface_quality',
    # cut
    'cut':'cut','cut_inner_insulation':'cut','cut_outer_insulation':'cut','cut_lead':'cut',
    # minor_defect
    'missing':'minor_defect','misplaced':'minor_defect',
    'poke_insulation':'minor_defect','poke':'minor_defect',
    'cable_swap':'minor_defect','combined':'minor_defect',
    'missing_cable':'minor_defect','thread':'minor_defect',
    'thread_side':'minor_defect','thread_top':'minor_defect',
    'pill_type':'minor_defect','manipulated_front':'minor_defect',
    'fabric_border':'minor_defect','fabric_interior':'minor_defect','defective':'minor_defect',
}

def map_defect(name):
    """Map a defect folder name to one of 8 semantic groups (fuzzy fallback)."""
    if name in DEFECT_GROUP_MAP:
        return DEFECT_GROUP_MAP[name]
    nl = name.lower().replace('_', '')
    for kw, grp in [('crack','fracture'),('break','fracture'),('split','fracture'),
                    ('hole','hole_void'),('void','hole_void'),('blowhole','hole_void'),
                    ('scratch','scratch'),('fray','scratch'),
                    ('bent','deformation'),('deform','deformation'),('squee','deformation'),
                    ('contam','contamination'),('glue','contamination'),('oil','contamination'),
                    ('color','surface_quality'),('rough','surface_quality'),('uneven','surface_quality'),
                    ('cut','cut')]:
        if kw in nl:
            return grp
    return 'minor_defect'  # safe fallback

# ── 17 PRODUCT CLASSES ────────────────────────────────────────────────────
MVTEC_CATS   = ['bottle','cable','capsule','carpet','grid','hazelnut',
                'leather','metal_nut','pill','screw','tile','toothbrush',
                'transistor','wood','zipper']
PRODUCT_LIST = MVTEC_CATS + ['casting', 'magnetic_tile']
PROD_TO_IDX  = {n: i for i, n in enumerate(PRODUCT_LIST)}

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'✅ CELL 1 done')
print(f'   device={device}')
print(f'   defect_types={NUM_DEFECT_TYPES}  product_types={NUM_PRODUCT_TYPES}')
print(f'   output → {OUTPUT_DIR}')
# Verify dataset paths exist
for label, p in [('MVTec', MVTEC_PATH),('Casting', CASTING_PATH),('Magnetic', MAGNETIC_PATH)]:
    status = '✅' if p.exists() else '❌ NOT FOUND — check path'
    print(f'   {label:10s} {status}')

✅ CELL 1 done
   device=cuda
   defect_types=8  product_types=17
   output → /home/sufi/training_results/ablation
   MVTec      ✅
   Casting    ✅
   Magnetic   ✅


In [2]:
# ════════════════════════════════════════════════════════════════════════
# CELL 2 — SCAN RAW DATASETS → DATAFRAME  (no merged CSV needed)
# ════════════════════════════════════════════════════════════════════════

def _imgs(folder):
    """Return all image paths inside a folder (jpg/jpeg/png/bmp)."""
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
    paths = []
    for ext in exts:
        paths.extend(folder.glob(ext))
    return paths


def scan_mvtec(base):
    recs = []
    for cat_dir in sorted(base.iterdir()):
        if not cat_dir.is_dir(): continue
        cat = cat_dir.name
        if cat not in PROD_TO_IDX: continue
        pid = PROD_TO_IDX[cat]
        # Normal: train/good + test/good
        for split in ['train', 'test']:
            gd = cat_dir / split / 'good'
            if gd.exists():
                for img in _imgs(gd):
                    recs.append({'path': str(img), 'binary': 0,
                                 'defect_type': -1, 'product': pid, 'source': 'mvtec'})
        # Defective: test/<defect_type>/
        test_dir = cat_dir / 'test'
        if test_dir.exists():
            for dd in sorted(test_dir.iterdir()):
                if dd.name == 'good' or not dd.is_dir(): continue
                grp = map_defect(dd.name)
                did = DEFECT_TO_IDX[grp]
                for img in _imgs(dd):
                    recs.append({'path': str(img), 'binary': 1,
                                 'defect_type': did, 'product': pid, 'source': 'mvtec'})
    return recs


def scan_casting(base):
    recs = []
    seen = set()
    pid  = PROD_TO_IDX['casting']
    # Walk all subdirectories to handle nested structure
    all_dirs = [base] + [d for d in base.rglob('*') if d.is_dir()]
    for d in all_dirs:
        if d.name == 'ok_front':
            for img in _imgs(d):
                s = str(img)
                if s not in seen:
                    seen.add(s)
                    recs.append({'path': s, 'binary': 0,
                                 'defect_type': -1, 'product': pid, 'source': 'casting'})
        elif d.name == 'def_front':
            did = DEFECT_TO_IDX['surface_quality']
            for img in _imgs(d):
                s = str(img)
                if s not in seen:
                    seen.add(s)
                    recs.append({'path': s, 'binary': 1,
                                 'defect_type': did, 'product': pid, 'source': 'casting'})
    return recs


def scan_magnetic(base):
    recs = []
    pid  = PROD_TO_IDX['magnetic_tile']
    for d in sorted(base.iterdir()):
        if not d.is_dir(): continue
        if d.name == 'MT_Free':
            for img in _imgs(d):
                recs.append({'path': str(img), 'binary': 0,
                             'defect_type': -1, 'product': pid, 'source': 'magnetic'})
        elif d.name.startswith('MT_'):
            grp = map_defect(d.name)
            did = DEFECT_TO_IDX[grp]
            for img in _imgs(d):
                recs.append({'path': str(img), 'binary': 1,
                             'defect_type': did, 'product': pid, 'source': 'magnetic'})
    return recs


print('Scanning datasets...')
t0 = time.time()

recs = []
recs += scan_mvtec(MVTEC_PATH);       print(f'  MVTec:   {len(recs):>6} imgs')
n = len(recs)
recs += scan_casting(CASTING_PATH);   print(f'  Casting: {len(recs)-n:>6} imgs')
n = len(recs)
recs += scan_magnetic(MAGNETIC_PATH); print(f'  Magnet:  {len(recs)-n:>6} imgs')
print(f'  TOTAL:   {len(recs):>6} imgs  ({time.time()-t0:.1f}s)')

full_df = pd.DataFrame(recs)
print(f'\nBinary split:  Normal={( full_df.binary==0).sum()}  Defective={(full_df.binary==1).sum()}')
print(f'Sources:  {dict(full_df.source.value_counts())}')
print(f'Defect groups: {dict(Counter(full_df[full_df.binary==1].defect_type.map({v:k for k,v in DEFECT_TO_IDX.items()})))}')
print('\n✅ CELL 2 done')

Scanning datasets...
  MVTec:     5354 imgs
  Casting:   8648 imgs
  Magnet:       0 imgs
  TOTAL:    14002 imgs  (0.2s)

Binary split:  Normal=7752  Defective=6250
Sources:  {'casting': 8648, 'mvtec': 5354}
Defect groups: {'fracture': 226, 'contamination': 162, 'deformation': 136, 'minor_defect': 320, 'cut': 87, 'scratch': 140, 'surface_quality': 5134, 'hole_void': 45}

✅ CELL 2 done


In [3]:
# ════════════════════════════════════════════════════════════════════════
# CELL 3 — TRAIN/VAL/TEST SPLIT + DATASET CLASS + DATALOADERS
# Splits are FIXED across all 7 runs (same seed + same full_df).
# ════════════════════════════════════════════════════════════════════════

# Stratification key: binary × source (keeps dataset proportions balanced)
strat_key = full_df['binary'].astype(str) + '_' + full_df['source']

# Step 1: 70% train vs 30% temp
train_df, temp_df = train_test_split(
    full_df, test_size=0.30, random_state=SEED, stratify=strat_key)

# Step 2: from 30% temp → 20% val + 10% test
temp_strat = temp_df['binary'].astype(str) + '_' + temp_df['source']
val_df, test_df = train_test_split(
    temp_df, test_size=1/3, random_state=SEED, stratify=temp_strat)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Split sizes: train={len(train_df)}  val={len(val_df)}  test={len(test_df)}')

# ── Transforms ────────────────────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD)
])

# ── Dataset class ─────────────────────────────────────────────────────────
class IndustrialDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return (
            img,
            torch.tensor(int(row['binary']),      dtype=torch.long),
            torch.tensor(int(row['defect_type']), dtype=torch.long),
            torch.tensor(int(row['product']),     dtype=torch.long),
            str(row.get('source', ''))
        )

# ── DataLoaders (built once, reused by all 7 runs) ─────────────────────────
train_ds = IndustrialDataset(train_df, train_tf)
val_ds   = IndustrialDataset(val_df,   val_tf)
test_ds  = IndustrialDataset(test_df,  val_tf)

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, generator=g)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True)

print(f'Loaders ready: {len(train_loader)} train batches | {len(val_loader)} val | {len(test_loader)} test')
print('✅ CELL 3 done')

Split sizes: train=9801  val=2800  test=1401
Loaders ready: 307 train batches | 44 val | 22 test
✅ CELL 3 done


In [4]:
# ════════════════════════════════════════════════════════════════════════
# CELL 4 — MODEL ARCHITECTURE
# Flags control which components are active:
#   use_b2       → EfficientNet-B2 backbone (else B0)
#   use_ca       → CoordAttention after backbone
#   use_ms_head  → Multi-scale defect head (else single-layer)
# ════════════════════════════════════════════════════════════════════════

class CoordAttention(nn.Module):
    """Position-aware channel attention (Hou et al. 2021)."""
    def __init__(self, inp, oup, reduction=32):
        super().__init__()
        mid = max(8, inp // reduction)
        self.pool_h  = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w  = nn.AdaptiveAvgPool2d((1, None))
        self.conv1   = nn.Conv2d(inp, mid, 1, bias=False)
        self.bn1     = nn.BatchNorm2d(mid)
        self.act     = nn.Hardswish()
        self.conv_h  = nn.Conv2d(mid, oup, 1, bias=False)
        self.conv_w  = nn.Conv2d(mid, oup, 1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.pool_h(x)                         # B×C×H×1
        w = self.pool_w(x).permute(0, 1, 3, 2)     # B×C×W×1
        y = torch.cat([h, w], dim=2)               # B×C×(H+W)×1
        y = self.act(self.bn1(self.conv1(y)))       # B×mid×(H+W)×1
        h_feat, w_feat = torch.split(y, [H, W], dim=2)
        h_attn = self.conv_h(h_feat).sigmoid()     # B×C×H×1
        w_attn = self.conv_w(w_feat).permute(0,1,3,2).sigmoid()  # B×C×1×W
        return x * h_attn * w_attn


class MultiScaleDefectHead(nn.Module):
    """Three-branch defect classifier fused at different compression levels."""
    def __init__(self, in_features, num_classes, drop=0.3):
        super().__init__()
        self.b1 = nn.Sequential(nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(drop))
        self.b2 = nn.Sequential(nn.Linear(in_features, 128), nn.ReLU(), nn.Dropout(drop))
        self.b3 = nn.Sequential(nn.Linear(in_features, 64),  nn.ReLU(), nn.Dropout(drop))
        self.fc = nn.Linear(256 + 128 + 64, num_classes)

    def forward(self, x):
        return self.fc(torch.cat([self.b1(x), self.b2(x), self.b3(x)], dim=1))


class AblationModel(nn.Module):
    def __init__(self, use_b2=False, use_ca=False, use_ms_head=False):
        super().__init__()
        bb_name = 'efficientnet_b2' if use_b2 else 'efficientnet_b0'
        # num_classes=0 removes the classifier; global_pool='' disables timm pooling
        self.backbone = timm.create_model(bb_name, pretrained=True,
                                          num_classes=0, global_pool='')
        feat_dim = self.backbone.num_features  # 1280 (B0) or 1408 (B2)

        self.use_ca = use_ca
        if use_ca:
            self.ca = CoordAttention(feat_dim, feat_dim)

        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, 512), nn.SiLU(), nn.Dropout(0.3)
        )
        self.binary_head  = nn.Linear(512, 1)
        self.product_head = nn.Linear(512, NUM_PRODUCT_TYPES)
        if use_ms_head:
            self.defect_head = MultiScaleDefectHead(512, NUM_DEFECT_TYPES)
        else:
            self.defect_head = nn.Sequential(
                nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(256, NUM_DEFECT_TYPES)
            )

    def forward(self, x):
        feat = self.backbone.forward_features(x)  # B×C×H×W
        if self.use_ca:
            feat = self.ca(feat)
        feat = self.gap(feat)     # B×C×1×1
        x    = self.shared(feat)  # B×512
        return self.binary_head(x), self.defect_head(x), self.product_head(x)


# Quick sanity check
with torch.no_grad():
    m = AblationModel(use_b2=True, use_ca=True, use_ms_head=True)
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
    b_out, d_out, p_out = m(dummy)
    print(f'Forward check: binary={tuple(b_out.shape)}  defect={tuple(d_out.shape)}  product={tuple(p_out.shape)}')
    print(f'Params (B2+CA+MS): {sum(p.numel() for p in m.parameters())/1e6:.2f}M')
del m
print('✅ CELL 4 done')

Forward check: binary=(2, 1)  defect=(2, 8)  product=(2, 17)
Params (B2+CA+MS): 8.85M
✅ CELL 4 done


In [5]:
# ════════════════════════════════════════════════════════════════════════
# CELL 5 — LOSSES + EMA + CUTMIX + METRICS
# ════════════════════════════════════════════════════════════════════════

# ── Focal Loss (ignores -1 labels) ────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=-1):
        super().__init__()
        self.gamma = gamma; self.weight = weight; self.ign = ignore_index

    def forward(self, logits, targets):
        mask = targets != self.ign
        if mask.sum() == 0: return logits.sum() * 0.0
        l, t = logits[mask], targets[mask]
        ce   = F.cross_entropy(l, t, weight=self.weight, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


# ── Learned uncertainty weighting (Kendall et al. 2018) ───────────────────
class UncertaintyWeighting(nn.Module):
    def __init__(self, n_tasks=3):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(n_tasks))

    def forward(self, losses):
        total = sum(torch.exp(-self.log_vars[i]) * L + 0.5 * self.log_vars[i]
                    for i, L in enumerate(losses))
        return total


# ── EMA ───────────────────────────────────────────────────────────────────
class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay  = decay
        self.shadow = {k: v.clone().float()
                       for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}

    def update(self, model):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(),
                                                      alpha=1 - self.decay)

    def apply(self, model):
        self._bak = {k: v.clone() for k, v in model.state_dict().items()}
        for k, v in model.state_dict().items():
            if k in self.shadow: v.copy_(self.shadow[k].to(v.dtype))

    def restore(self, model):
        for k, v in model.state_dict().items(): v.copy_(self._bak[k])
        del self._bak


# ── CutMix ────────────────────────────────────────────────────────────────
def cutmix_batch(imgs, alpha=1.0):
    """Mix images only (labels stay original — simpler, avoids ignore_index issues)."""
    B = imgs.size(0)
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(B, device=imgs.device)
    W, H = imgs.size(3), imgs.size(2)
    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx, cy = random.randint(0, W), random.randint(0, H)
    x1, x2 = max(0, cx - cut_w // 2), min(W, cx + cut_w // 2)
    y1, y2 = max(0, cy - cut_h // 2), min(H, cy + cut_h // 2)
    mixed  = imgs.clone()
    mixed[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
    return mixed


# ── Evaluation ────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, ema=None):
    if ema is not None: ema.apply(model)
    model.eval()
    bin_p, bin_t, def_p, def_t, prod_p, prod_t = [], [], [], [], [], []

    for imgs, bl, dl, pl, _ in loader:
        imgs = imgs.to(device)
        with autocast('cuda', enabled=torch.cuda.is_available()):
            b_out, d_out, p_out = model(imgs)
        bin_p.extend((torch.sigmoid(b_out.squeeze(1)) > 0.5).long().cpu().numpy())
        bin_t.extend(bl.numpy())
        def_p.extend(d_out.argmax(1).cpu().numpy())
        def_t.extend(dl.numpy())
        prod_p.extend(p_out.argmax(1).cpu().numpy())
        prod_t.extend(pl.numpy())

    if ema is not None: ema.restore(model)

    bp, bt = np.array(bin_p),  np.array(bin_t)
    dp, dt = np.array(def_p),  np.array(def_t)
    pp, pt = np.array(prod_p), np.array(prod_t)

    binary_acc  = 100.0 * (bp == bt).mean()
    product_acc = 100.0 * (pp == pt).mean()

    mask = dt != -1
    if mask.sum() > 0:
        _, _, per_f1, _ = precision_recall_fscore_support(
            dt[mask], dp[mask],
            labels=list(range(NUM_DEFECT_TYPES)),
            average=None, zero_division=0)
        macro_f1   = float(per_f1.mean())
        defect_acc = 100.0 * (dp[mask] == dt[mask]).mean()
    else:
        per_f1 = np.zeros(NUM_DEFECT_TYPES)
        macro_f1 = defect_acc = 0.0

    return {'macro_f1': macro_f1, 'defect_acc': float(defect_acc),
            'binary_acc': float(binary_acc), 'product_acc': float(product_acc),
            'per_class_f1': per_f1.tolist()}


@torch.no_grad()
def measure_latency(model, n_warm=30, n_meas=200):
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    for _ in range(n_warm): _ = model(x)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_meas): _ = model(x)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ms = (time.time() - t0) / n_meas * 1000
    return ms, 1000.0 / ms


print('✅ CELL 5 done — FocalLoss, UncertaintyWeighting, EMA, CutMix, evaluate, measure_latency')

✅ CELL 5 done — FocalLoss, UncertaintyWeighting, EMA, CutMix, evaluate, measure_latency


In [6]:
# ════════════════════════════════════════════════════════════════════════
# CELL 6 — SINGLE-RUN TRAINER
# • Loads existing JSON and skips if run already done (auto-skip).
# • Saves result JSON + checkpoint immediately after finishing.
# • Cleans up GPU memory after each run.
# ════════════════════════════════════════════════════════════════════════

def train_one_config(cfg, verbose=True):
    name      = cfg['name']
    json_path = RESULTS_DIR / f'{name}.json'
    ckpt_path = CKPT_DIR    / f'{name}.pth'

    # ── AUTO-SKIP ──────────────────────────────────────────────────────────
    if json_path.exists():
        print(f'⏭️  SKIP {name}  (found {json_path.name})')
        return json.loads(json_path.read_text())

    print(f'\n{"="*65}')
    print(f'  ▶ {name}')
    for k, v in cfg.items():
        if k != 'name': print(f'      {k}: {v}')
    print(f'{"="*65}')

    set_seed()  # always reset before each run

    # ── BUILD MODEL ────────────────────────────────────────────────────────
    model = AblationModel(
        use_b2      = cfg.get('use_b2',      False),
        use_ca      = cfg.get('use_ca',       False),
        use_ms_head = cfg.get('use_ms_head',  False),
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # ── LOSSES ─────────────────────────────────────────────────────────────
    criterion_binary  = nn.BCEWithLogitsLoss()
    criterion_product = nn.CrossEntropyLoss()

    if cfg.get('use_focal', False):
        # compute per-class frequency weights
        cnt = Counter(train_df[train_df.defect_type != -1]['defect_type'].values)
        total = sum(cnt.values())
        w = torch.tensor([total / (NUM_DEFECT_TYPES * cnt.get(i, 1))
                           for i in range(NUM_DEFECT_TYPES)], dtype=torch.float, device=device)
        w = w / w.sum() * NUM_DEFECT_TYPES   # normalise
        criterion_defect = FocalLoss(gamma=2.0, weight=w)
    else:
        criterion_defect = nn.CrossEntropyLoss(ignore_index=-1)

    # ── UNCERTAINTY WEIGHTING ──────────────────────────────────────────────
    unc = None
    if cfg.get('use_uncertainty', False):
        unc = UncertaintyWeighting(n_tasks=3).to(device)
        opt_params = list(model.parameters()) + list(unc.parameters())
    else:
        opt_params = model.parameters()

    # ── OPTIMIZER ─────────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(opt_params, lr=1e-4, weight_decay=1e-4)

    # ── SCHEDULER ─────────────────────────────────────────────────────────
    if cfg.get('use_warm_restart', False):
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=25, T_mult=1, eta_min=1e-6)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS, eta_min=1e-6)

    # ── EMA ───────────────────────────────────────────────────────────────
    ema = EMA(model, decay=0.9995) if cfg.get('use_ema', False) else None

    scaler = GradScaler('cuda') if torch.cuda.is_available() else None
    best_f1, best_ckpt = 0.0, None

    # ── TRAINING LOOP ─────────────────────────────────────────────────────
    for epoch in range(1, EPOCHS + 1):
        model.train()
        if unc: unc.train()
        epoch_loss = 0.0; n_b = 0

        for imgs, bl, dl, pl, _ in train_loader:
            imgs = imgs.to(device)
            bl   = bl.float().to(device)
            dl   = dl.to(device)
            pl   = pl.to(device)

            if cfg.get('use_cutmix', False) and random.random() < 0.5:
                imgs = cutmix_batch(imgs)

            optimizer.zero_grad()

            if scaler:
                with autocast('cuda'):
                    b_out, d_out, p_out = model(imgs)
                    l_bin  = criterion_binary(b_out.squeeze(1), bl)
                    l_def  = criterion_defect(d_out, dl)
                    l_prod = criterion_product(p_out, pl)
                    loss   = unc([l_bin, l_def, l_prod]) if unc else \
                             0.2 * l_bin + 0.6 * l_def + 0.2 * l_prod
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer); scaler.update()
            else:
                b_out, d_out, p_out = model(imgs)
                l_bin  = criterion_binary(b_out.squeeze(1), bl)
                l_def  = criterion_defect(d_out, dl)
                l_prod = criterion_product(p_out, pl)
                loss   = unc([l_bin, l_def, l_prod]) if unc else \
                         0.2 * l_bin + 0.6 * l_def + 0.2 * l_prod
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            if ema: ema.update(model)
            epoch_loss += loss.item(); n_b += 1

        scheduler.step()

        # Evaluate every 5 epochs and at end
        if epoch % 5 == 0 or epoch == EPOCHS:
            m = evaluate(model, val_loader, ema=ema)
            if m['macro_f1'] > best_f1:
                best_f1 = m['macro_f1']
                torch.save(model.state_dict(), ckpt_path)
            if verbose:
                print(f'  ep {epoch:3d} | loss={epoch_loss/n_b:.4f} | '
                      f'F1={m["macro_f1"]:.4f} | '
                      f'DefAcc={m["defect_acc"]:.2f}% | '
                      f'BinAcc={m["binary_acc"]:.2f}%')

    # ── FINAL TEST EVAL WITH BEST WEIGHTS ─────────────────────────────────
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    test_m = evaluate(model, test_loader)
    lat_ms, fps = measure_latency(model)

    result = {
        'name':        name,
        'macro_f1':    test_m['macro_f1'],
        'defect_acc':  test_m['defect_acc'],
        'binary_acc':  test_m['binary_acc'],
        'product_acc': test_m['product_acc'],
        'per_class_f1':test_m['per_class_f1'],
        'params_M':    n_params / 1e6,
        'latency_ms':  lat_ms,
        'fps':         fps,
    }
    json_path.write_text(json.dumps(result, indent=2))
    print(f'\n  ✅ {name}  F1={result["macro_f1"]:.4f}  '
          f'DefAcc={result["defect_acc"]:.2f}%  '
          f'Params={result["params_M"]:.2f}M')
    print(f'     saved → {json_path}')

    # Clean up VRAM before next run
    del model
    if unc: del unc
    gc.collect(); torch.cuda.empty_cache()

    return result


print('✅ CELL 6 done — train_one_config() defined')

✅ CELL 6 done — train_one_config() defined


In [7]:
# ════════════════════════════════════════════════════════════════════════
# CELL 7 — 7 ABLATION CONFIGS  (cumulative build-up)
# ════════════════════════════════════════════════════════════════════════

ABLATION_CONFIGS = [
    # ── Config 1: Baseline B0 + MTL ──────────────────────────────────────
    {'name': 'R1_B0_MTL',
     'use_b2': False, 'use_ca': False, 'use_ms_head': False,
     'use_focal': False, 'use_warm_restart': False,
     'use_cutmix': False, 'use_uncertainty': False, 'use_ema': False},

    # ── Config 2: + B2 backbone ──────────────────────────────────────────
    {'name': 'R2_B0_to_B2',
     'use_b2': True,  'use_ca': False, 'use_ms_head': False,
     'use_focal': False, 'use_warm_restart': False,
     'use_cutmix': False, 'use_uncertainty': False, 'use_ema': False},

    # ── Config 3: + CoordAttention ───────────────────────────────────────
    {'name': 'R3_B2_CoordAttn',
     'use_b2': True,  'use_ca': True,  'use_ms_head': False,
     'use_focal': False, 'use_warm_restart': False,
     'use_cutmix': False, 'use_uncertainty': False, 'use_ema': False},

    # ── Config 4: + Multi-scale defect head ──────────────────────────────
    {'name': 'R4_B2_CA_MSHead',
     'use_b2': True,  'use_ca': True,  'use_ms_head': True,
     'use_focal': False, 'use_warm_restart': False,
     'use_cutmix': False, 'use_uncertainty': False, 'use_ema': False},

    # ── Config 5: + Weighted Focal Loss ──────────────────────────────────
    {'name': 'R5_B2_CA_MSHead_Focal',
     'use_b2': True,  'use_ca': True,  'use_ms_head': True,
     'use_focal': True, 'use_warm_restart': False,
     'use_cutmix': False, 'use_uncertainty': False, 'use_ema': False},

    # ── Config 6: + Warm Restarts ─────────────────────────────────────────
    {'name': 'R6_B2_CA_MSHead_Focal_WR',
     'use_b2': True,  'use_ca': True,  'use_ms_head': True,
     'use_focal': True, 'use_warm_restart': True,
     'use_cutmix': False, 'use_uncertainty': False, 'use_ema': False},

    # ── Config 7: Full (+ CutMix + UncertaintyWeighting + EMA) ───────────
    {'name': 'R7_Full_CutMix_Unc_EMA',
     'use_b2': True,  'use_ca': True,  'use_ms_head': True,
     'use_focal': True, 'use_warm_restart': True,
     'use_cutmix': True, 'use_uncertainty': True, 'use_ema': True},
]

print('7 ablation configs:')
for i, c in enumerate(ABLATION_CONFIGS, 1):
    flags = [k.replace('use_','') for k, v in c.items() if k.startswith('use_') and v]
    print(f'  {i}. {c["name"]:35s}  [{", ".join(flags) if flags else "baseline"}]')
print('\n✅ CELL 7 done')

7 ablation configs:
  1. R1_B0_MTL                            [baseline]
  2. R2_B0_to_B2                          [b2]
  3. R3_B2_CoordAttn                      [b2, ca]
  4. R4_B2_CA_MSHead                      [b2, ca, ms_head]
  5. R5_B2_CA_MSHead_Focal                [b2, ca, ms_head, focal]
  6. R6_B2_CA_MSHead_Focal_WR             [b2, ca, ms_head, focal, warm_restart]
  7. R7_Full_CutMix_Unc_EMA               [b2, ca, ms_head, focal, warm_restart, cutmix, uncertainty, ema]

✅ CELL 7 done


In [8]:
# ════════════════════════════════════════════════════════════════════════
# CELL 8 — RUN LOOP   (this is the long-running cell)
#
# RUN_IDX options:
#   'all'   → run all 7 in sequence  (~3.5–4.5 hrs on RTX 3060)
#   0       → run only Config 1
#   1       → run only Config 2
#   ...     → etc.
#
# Completed runs are SAVED immediately and SKIPPED on re-run.
# Safe to interrupt and re-run — picks up from where you left off.
# ════════════════════════════════════════════════════════════════════════

RUN_IDX = 2   # ← CHANGE THIS to 0/1/2/3/4/5/6 to run one at a time

results_so_far = []

if RUN_IDX == 'all':
    print(f'Running all 7 configs. Estimated time: ~3.5–4.5 hrs on RTX 3060.')
    print(f'Completed runs are auto-skipped.\n')
    for i, cfg in enumerate(ABLATION_CONFIGS):
        r = train_one_config(cfg)
        results_so_far.append(r)
    print('\n🎉 All 7 configs done!')
else:
    idx = int(RUN_IDX)
    assert 0 <= idx <= 6, 'RUN_IDX must be 0–6 or "all"'
    print(f'Running config {idx+1}/7: {ABLATION_CONFIGS[idx]["name"]}')
    r = train_one_config(ABLATION_CONFIGS[idx])
    results_so_far.append(r)

Running config 3/7: R3_B2_CoordAttn

  ▶ R3_B2_CoordAttn
      use_b2: True
      use_ca: True
      use_ms_head: False
      use_focal: False
      use_warm_restart: False
      use_cutmix: False
      use_uncertainty: False
      use_ema: False


  ep   5 | loss=0.1699 | F1=0.5148 | DefAcc=90.40% | BinAcc=94.32%
  ep  10 | loss=0.0858 | F1=0.7431 | DefAcc=94.32% | BinAcc=96.96%
  ep  15 | loss=0.0511 | F1=0.8511 | DefAcc=96.88% | BinAcc=98.18%
  ep  20 | loss=0.0255 | F1=0.8620 | DefAcc=97.12% | BinAcc=98.61%
  ep  25 | loss=0.0181 | F1=0.8781 | DefAcc=97.36% | BinAcc=98.86%


KeyboardInterrupt: 

In [9]:
# ======================================================================
# CELL 4 — EdgeNetV4  (EfficientNet-B2 pretrained backbone)
# NO COORDINATE ATTENTION VERSION
# ======================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from pathlib import Path

# ── Save path ───────────────────────────────────────────────────────────
V4_DIR  = Path('/home/sufi/training_results/models/V4')
V4_DIR.mkdir(parents=True, exist_ok=True)
V4_SAVE = V4_DIR / 'EdgeNet_V4_best.pth'


# ── Model ────────────────────────────────────────────────────────────────
class EdgeNetV4(nn.Module):

    def __init__(self, num_defect_classes=8, num_product_classes=17):
        super().__init__()

        # ── EfficientNet-B2 backbone ─────────────────────────────────
        self.backbone = timm.create_model(
            'efficientnet_b2',
            pretrained=True,
            features_only=True,
            out_indices=(2, 3, 4),
        )

        chs = self.backbone.feature_info.channels()  # [48, 120, 352]
        c2, c3, c4 = chs[0], chs[1], chs[2]

        # ── Shared semantic neck (deepest feature) ───────────────────
        self.neck = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(c4, 512, bias=False),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.35),
        )

        # ── Binary head ───────────────────────────────────────────────
        self.binary_head = nn.Linear(512, 1)

        # ── Product head ──────────────────────────────────────────────
        self.product_head = nn.Sequential(
            nn.Linear(512, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_product_classes),
        )

        # ── Multi-scale defect head ───────────────────────────────────
        self.defect_gap = nn.AdaptiveAvgPool2d(1)

        defect_in = c2 + c3 + c4  # 48 + 120 + 352 = 520

        self.defect_head = nn.Sequential(
            nn.Linear(defect_in, 384, bias=False),
            nn.BatchNorm1d(384),
            nn.SiLU(),
            nn.Dropout(0.3),

            nn.Linear(384, 192, bias=False),
            nn.BatchNorm1d(192),
            nn.SiLU(),
            nn.Dropout(0.2),

            nn.Linear(192, num_defect_classes),
        )

        self._init_heads()

    def _init_heads(self):
        for module in [
            self.neck,
            self.binary_head,
            self.product_head,
            self.defect_head
        ]:
            for layer in module.modules():

                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)

                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)

                elif isinstance(layer, (nn.BatchNorm1d, nn.BatchNorm2d)):
                    nn.init.ones_(layer.weight)
                    nn.init.zeros_(layer.bias)

    def forward(self, x):

        # Multi-scale backbone features
        f2, f3, f4 = self.backbone(x)

        # Shared semantic descriptor
        shared = self.neck(f4)

        # Binary + product
        bin_out  = self.binary_head(shared)
        prod_out = self.product_head(shared)

        # Multi-scale defect descriptor
        d2 = self.defect_gap(f2).flatten(1)
        d3 = self.defect_gap(f3).flatten(1)
        d4 = self.defect_gap(f4).flatten(1)

        def_feat = torch.cat([d2, d3, d4], dim=1)
        def_out  = self.defect_head(def_feat)

        return bin_out, def_out, prod_out

    def count_params(self):
        total = sum(
            p.numel()
            for p in self.parameters()
            if p.requires_grad
        )
        return total / 1e6


# ══════════════════════════════════════════════════════════════════════
# Build + sanity checks
# ══════════════════════════════════════════════════════════════════════

model = EdgeNetV4(
    num_defect_classes=NUM_DEFECT_TYPES,
    num_product_classes=NUM_PRODUCT_TYPES,
).to(device)

model.eval()

with torch.no_grad():
    _dummy = torch.randn(2, 3, 224, 224).to(device)

    _b, _d, _p = model(_dummy)

    assert _b.shape == (2, 1), \
        f"binary wrong  : {_b.shape}"

    assert _d.shape == (2, NUM_DEFECT_TYPES), \
        f"defect wrong  : {_d.shape}"

    assert _p.shape == (2, NUM_PRODUCT_TYPES), \
        f"product wrong : {_p.shape}"

    del _dummy, _b, _d, _p

_chs = model.backbone.feature_info.channels()
_defect_in_sz = _chs[0] + _chs[1] + _chs[2]

print("✅ CELL 4 COMPLETE — EdgeNetV4 (NO COORD ATTENTION)")
print("   Backbone   : EfficientNet-B2 pretrained=True (timm)")
print(f"   Features   : {_chs[0]}ch@s8 + {_chs[1]}ch@s16 + {_chs[2]}ch@s32")
print(
    f"   Defect in  : {_chs[0]}+{_chs[1]}+{_chs[2]} = "
    f"{_defect_in_sz} → 384 → 192 → {NUM_DEFECT_TYPES}"
)
print(f"   Total params : {model.count_params():.2f}M")
print(f"   Save path  : {V4_SAVE}")

Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


✅ CELL 4 COMPLETE — EdgeNetV4 (NO COORD ATTENTION)
   Backbone   : EfficientNet-B2 pretrained=True (timm)
   Features   : 48ch@s8 + 120ch@s16 + 352ch@s32
   Defect in  : 48+120+352 = 520 → 384 → 192 → 8
   Total params : 7.80M
   Save path  : /home/sufi/training_results/models/V4/EdgeNet_V4_best.pth


In [10]:
# ======================================================================
# CELL 5 — EdgeNetV4 Training  (EfficientNet-B2 backbone, v4)
# ======================================================================
# Requires from previous cells:
#   Cell 1 : device, NUM_DEFECT_TYPES, NUM_PRODUCT_TYPES, SEMANTIC_GROUPS
#             DEFECT_TYPE_TO_IDX, PRODUCT_TYPE_TO_IDX
#             IDX_TO_DEFECT_TYPE, IDX_TO_PRODUCT_TYPE
#   Cell 2B: criterion_dict   (binary, product, multitask; defect overridden)
#   Cell 3 : train_3head_loader, val_3head_loader
#   Cell 4 : model, V4_DIR, V4_SAVE
#
# Key fixes over v3:
#   FIX 1 — Backbone LR was always ~0.
#     Root cause: make_optimizer(backbone_lr=0.0) caused the scheduler to
#     permanently track initial_lr=0. Even after the manual
#     optimizer.param_groups[0]['lr']=LR_BACKBONE at epoch 4, the very next
#     scheduler.step() overrode it back toward zero.
#     Fix: create optimizer with backbone_lr=LR_BACKBONE from the start.
#     Freeze backbone using requires_grad=False only. When unfreezing,
#     just set requires_grad=True — the scheduler already tracks the
#     correct peak LR and continues its cosine curve without interruption.
#
#   FIX 2 — scratch/deformation recall stuck at 0.07-0.20.
#     Root cause: plain FocalLoss(gamma=2) with no class weights.
#     minor_defect is ~10x more common than deformation; focal modulation
#     alone cannot overcome that frequency gap when the backbone isn't
#     fine-tuning either.
#     Fix: compute inverse-frequency class weights from training data
#     and apply them as a multiplicative factor OUTSIDE the focal
#     modulation (correct formulation; avoids corrupting pt estimates).
# ======================================================================

import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from sklearn.metrics import precision_recall_fscore_support
from pathlib import Path

# ── Config ──────────────────────────────────────────────────────────────
EPOCHS        = 100
FREEZE_EPOCHS = 3       # epochs with backbone frozen (Phase 1)
LR_BACKBONE   = 1e-4    # fine-tune pretrained weights gently
LR_CA         = 5e-4    # CoordAttention — slightly higher than backbone
LR_HEAD       = 1e-3    # randomly-initialised heads — full LR
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
EMA_DECAY     = 0.9995
PATIENCE      = 35
EMA_START     = 5
FOCAL_GAMMA   = 2.0
MONITOR       = 'defect_f1'
METRICS_JSON  = V4_DIR / 'training_metrics.json'

# ── Class weights (computed from training data) ──────────────────────────
# Inverse-frequency weighting: rare classes get higher weight.
# Computed once before training; stored on device for use in FocalLoss.
print("  Computing inverse-frequency class weights from training data...")
_cls_counts = torch.zeros(NUM_DEFECT_TYPES)
with torch.no_grad():
    for _, _, _def_lbl, _, _ in train_3head_loader:
        _valid = _def_lbl[_def_lbl != -1]
        for _i in range(NUM_DEFECT_TYPES):
            _cls_counts[_i] += (_valid == _i).sum()

# w_i = total_defect_samples / (C * n_i), then normalise to mean=1
_total = _cls_counts.sum()
_cls_weights = _total / (_cls_counts.clamp(min=1) * NUM_DEFECT_TYPES)
_cls_weights = (_cls_weights / _cls_weights.mean()).to(device)

print(f"  Class counts : { {SEMANTIC_GROUPS[i]: int(_cls_counts[i].item()) for i in range(NUM_DEFECT_TYPES)} }")
print(f"  Class weights: { {SEMANTIC_GROUPS[i]: f'{_cls_weights[i].item():.2f}' for i in range(NUM_DEFECT_TYPES)} }")

# ── Focal Loss with separate class-weight multiplier ────────────────────
# Correct formulation: focal modulation (1-pt)^gamma uses UNweighted CE so
# that pt = model confidence, not a weighted pseudo-probability.
# Class weight is then applied as a multiplicative scalar per sample.
# This avoids distorting pt estimates while still boosting rare-class loss.

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ignore_index=-1):
        super().__init__()
        self.gamma        = gamma
        self.weight       = weight   # [C] float tensor on device, or None
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        valid = targets != self.ignore_index
        if valid.sum() == 0:
            return logits.sum() * 0.0
        t = targets[valid]
        l = logits[valid]

        # Unweighted CE → pt stays honest (= model's predicted probability)
        ce_unweighted = F.cross_entropy(l, t, reduction='none')
        pt = torch.exp(-ce_unweighted)
        focal_factor = (1.0 - pt) ** self.gamma

        # Apply class weights as a per-sample multiplier
        if self.weight is not None:
            cls_w = self.weight[t]        # [N]
            return (focal_factor * cls_w * ce_unweighted).mean()
        return (focal_factor * ce_unweighted).mean()

focal_defect = FocalLoss(gamma=FOCAL_GAMMA,
                         weight=_cls_weights,
                         ignore_index=-1).to(device)

# ── Parameter groups ────────────────────────────────────────────────────
# FIX 1: Always create optimizer with backbone_lr=LR_BACKBONE.
# Freeze backbone via requires_grad=False — the LR is tracked correctly
# by the scheduler; no gradients flow through frozen params regardless.
# At unfreeze we only call requires_grad=True; no LR patching needed.

def make_optimizer(backbone_lr=LR_BACKBONE):
    backbone_params = list(model.backbone.parameters())
    ca_params       = (list(model.ca_mid.parameters()) +
                       list(model.ca_deep.parameters()))
    head_params     = (list(model.neck.parameters())         +
                       list(model.binary_head.parameters())  +
                       list(model.product_head.parameters()) +
                       list(model.defect_head.parameters())  +
                       list(model.defect_gap.parameters() if
                            hasattr(model, 'defect_gap') else []) +
                       list(criterion_dict['multitask'].parameters()))
    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr},
        {'params': ca_params,       'lr': LR_CA},
        {'params': head_params,     'lr': LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

# Freeze backbone by stopping gradients — NOT by zeroing LR
for p in model.backbone.parameters():
    p.requires_grad = False

# Scheduler now tracks LR_BACKBONE as the backbone group's peak LR
optimizer = make_optimizer(backbone_lr=LR_BACKBONE)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=30, T_mult=2, eta_min=1e-6)

scaler = GradScaler('cuda')

# ── EMA ─────────────────────────────────────────────────────────────────
class EMA:
    def __init__(self, model, decay=0.9995):
        self.decay  = decay
        self.shadow = {k: v.clone().detach()
                       for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(
                    v.detach(), alpha=1.0 - self.decay)

    def apply(self, model):
        self._backup = {k: v.clone() for k, v in model.state_dict().items()}
        for k, v in model.state_dict().items():
            v.copy_(self.shadow[k])

    def restore(self, model):
        for k, v in model.state_dict().items():
            v.copy_(self._backup[k])
        del self._backup

ema = EMA(model, EMA_DECAY)

# ── Metrics helper ───────────────────────────────────────────────────────
def compute_defect_metrics(preds, labels):
    mask = labels != -1
    if mask.sum() == 0:
        z = np.zeros(NUM_DEFECT_TYPES)
        return {'f1': 0.0, 'prec': 0.0, 'rec': 0.0,
                'per_p': z, 'per_r': z, 'per_f': z}
    p, r, f, _ = precision_recall_fscore_support(
        labels[mask], preds[mask], average=None,
        labels=list(range(NUM_DEFECT_TYPES)), zero_division=0)
    return {'f1': float(f.mean()), 'prec': float(p.mean()),
            'rec': float(r.mean()),
            'per_p': p, 'per_r': r, 'per_f': f}

# ── Train one epoch ──────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scaler, ema, epoch):
    model.train()
    criterion_dict['multitask'].set_epoch(epoch)
    tot = bin_c = def_c = def_tot = prod_c = 0
    all_pred, all_true = [], []

    for imgs, bin_lbl, def_lbl, prod_lbl, _ in loader:
        imgs     = imgs.to(device)
        bin_lbl  = bin_lbl.to(device)
        def_lbl  = def_lbl.to(device)
        prod_lbl = prod_lbl.to(device)

        optimizer.zero_grad()

        with autocast('cuda'):
            sb, sd, sp = model(imgs)
            sb_sq  = sb.squeeze(1)
            l_bin  = criterion_dict['binary'](sb_sq, bin_lbl.float())
            l_def  = focal_defect(sd, def_lbl)
            l_prod = criterion_dict['product'](sp, prod_lbl)
            tw     = criterion_dict['multitask'].weights()
            l_tot  = (float(tw[0]) * l_bin  +
                      float(tw[1]) * l_def  +
                      float(tw[2]) * l_prod)

        scaler.scale(l_tot).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        tot += imgs.size(0)
        with torch.no_grad():
            bin_c  += ((torch.sigmoid(sb_sq) > 0.5).long() == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            preds   = sd.argmax(1)
            mask    = def_lbl != -1
            if mask.sum() > 0:
                def_c   += (preds[mask] == def_lbl[mask]).sum().item()
                def_tot += mask.sum().item()
            all_pred.extend(preds.cpu().numpy())
            all_true.extend(def_lbl.cpu().numpy())

    m = compute_defect_metrics(np.array(all_pred), np.array(all_true))
    return {
        'binary_acc':  100. * bin_c  / tot,
        'defect_acc':  100. * def_c  / max(def_tot, 1),
        'product_acc': 100. * prod_c / tot,
        'train_f1':    m['f1'],
        'train_per_p': m['per_p'],
        'train_per_r': m['per_r'],
        'train_per_f': m['per_f'],
    }

# ── Validate ─────────────────────────────────────────────────────────────
def validate_epoch(model, loader, ema, epoch):
    use_ema = epoch >= EMA_START
    if use_ema:
        ema.apply(model)
    model.eval()
    tot = bin_c = def_c = def_tot = prod_c = 0
    all_pred, all_true = [], []

    with torch.no_grad():
        for imgs, bin_lbl, def_lbl, prod_lbl, _ in loader:
            imgs     = imgs.to(device)
            bin_lbl  = bin_lbl.to(device)
            def_lbl  = def_lbl.to(device)
            prod_lbl = prod_lbl.to(device)
            with autocast('cuda'):
                sb, sd, sp = model(imgs)
                sb_sq      = sb.squeeze(1)
            tot    += imgs.size(0)
            bin_c  += ((torch.sigmoid(sb_sq) > 0.5).long() == bin_lbl).sum().item()
            prod_c += (sp.argmax(1) == prod_lbl).sum().item()
            preds   = sd.argmax(1)
            mask    = def_lbl != -1
            if mask.sum() > 0:
                def_c   += (preds[mask] == def_lbl[mask]).sum().item()
                def_tot += mask.sum().item()
            all_pred.extend(preds.cpu().numpy())
            all_true.extend(def_lbl.cpu().numpy())

    if use_ema:
        ema.restore(model)

    m = compute_defect_metrics(np.array(all_pred), np.array(all_true))
    return {
        'binary_acc':  100. * bin_c  / tot,
        'defect_acc':  100. * def_c  / max(def_tot, 1),
        'product_acc': 100. * prod_c / tot,
        'val_f1':      m['f1'],
        'val_per_p':   m['per_p'],
        'val_per_r':   m['per_r'],
        'val_per_f':   m['per_f'],
    }

# ── Training loop ─────────────────────────────────────────────────────────
history    = []
best_score = 0.0
no_improve = 0
prev_per_f = np.zeros(NUM_DEFECT_TYPES)
backbone_unfrozen = False

print(f"\n🚀 Training EdgeNetV4  ({model.count_params():.2f}M params)")
print(f"   Backbone  : EfficientNet-B2 pretrained (frozen for {FREEZE_EPOCHS} ep)")
print(f"   Patience  : {PATIENCE}   Monitor : {MONITOR}   Focal gamma={FOCAL_GAMMA}")
print(f"   Scheduler : CosineWarmRestarts T0=30 Tmult=2  (restarts @ ep 30,90,210)")
print(f"   LR        : backbone={LR_BACKBONE}  CA={LR_CA}  heads={LR_HEAD}")
print(f"   Save      : {V4_SAVE}")
print()

t0 = time.time()
for epoch in range(1, EPOCHS + 1):

    # ── Phase transition: unfreeze backbone after FREEZE_EPOCHS ─────────
    # FIX 1: simply re-enable gradients. Scheduler already tracks the
    # correct LR_BACKBONE peak — no manual LR override needed.
    if epoch == FREEZE_EPOCHS + 1 and not backbone_unfrozen:
        for p in model.backbone.parameters():
            p.requires_grad = True
        backbone_unfrozen = True
        bb_lr_now = optimizer.param_groups[0]['lr']
        print(f"  🔓 Epoch {epoch}: backbone UNFROZEN → current LR={bb_lr_now:.2e}")

    # ── EMA warm restart: flush random-init memory at first EMA epoch ────
    if epoch == EMA_START:
        ema.shadow = {k: v.clone().detach()
                      for k, v in model.state_dict().items()}

    tr = train_one_epoch(model, train_3head_loader, optimizer, scaler, ema, epoch)
    vl = validate_epoch(model, val_3head_loader, ema, epoch)
    scheduler.step()

    score  = vl['val_f1'] if MONITOR == 'defect_f1' else vl['defect_acc']
    per_p  = vl['val_per_p']
    per_r  = vl['val_per_r']
    per_f  = vl['val_per_f']
    tw     = criterion_dict['multitask'].weights()
    lrs    = [f"{pg['lr']:.1e}" for pg in optimizer.param_groups]
    beat   = "  🏆 BEATS TEACHER!" if vl['defect_acc'] > 93.92 else ""
    phase  = "[frozen]" if epoch <= FREEZE_EPOCHS else "[full  ]"
    r_tag  = "  🔄 LR RESTART" if epoch in (30, 90, 210) else ""

    print(f"\nEpoch [{epoch:>3}/{EPOCHS}] {phase}  "
          f"LR={lrs}  "
          f"TaskW=[{' '.join(f'{float(w):.2f}' for w in tw)}]"
          f"{beat}{r_tag}")
    print()
    print(f"  Train → Bin={tr['binary_acc']:.1f}%  "
          f"Def={tr['defect_acc']:.1f}%  "
          f"F1={tr['train_f1']:.3f}  "
          f"Prod={tr['product_acc']:.1f}%")
    print(f"  Val   → Bin={vl['binary_acc']:.1f}%  "
          f"Def={vl['defect_acc']:.1f}%  "
          f"F1={vl['val_f1']:.3f}  "
          f"Prod={vl['product_acc']:.1f}%")
    print()
    print(f"  {'Class':<22} {'P':>6}  {'R':>6}  {'F1':>6}  Trend")
    print(f"  {'─'*50}")
    for i, name in enumerate(SEMANTIC_GROUPS):
        delta = per_f[i] - prev_per_f[i]
        arrow = '↑' if delta > 0.005 else ('↓' if delta < -0.005 else '→')
        watch = '  ◄ WATCH' if per_r[i] < 0.50 else ''
        print(f"  {name:<22} {per_p[i]:.3f}  {per_r[i]:.3f}  "
              f"{per_f[i]:.3f}  {arrow}{watch}")
    print(f"  {'─'*50}")
    print(f"  {'MACRO':<22} {per_p.mean():.3f}  "
          f"{per_r.mean():.3f}  {per_f.mean():.3f}")
    prev_per_f = per_f.copy()

    # ── Checkpoint ──────────────────────────────────────────────────────
    if score > best_score:
        best_score = score
        no_improve = 0
        if epoch >= EMA_START:
            ema.apply(model)
        torch.save({
            'epoch':            epoch,
            'score':            best_score,
            'model_state_dict': model.state_dict(),
            'optimizer':        optimizer.state_dict(),
            'scheduler':        scheduler.state_dict(),
            'val_metrics': {
                'binary_acc':      vl['binary_acc'],
                'defect_acc':      vl['defect_acc'],
                'defect_f1_macro': vl['val_f1'],
                'product_acc':     vl['product_acc'],
            },
            'defect_type_to_idx':  DEFECT_TYPE_TO_IDX,
            'product_type_to_idx': PRODUCT_TYPE_TO_IDX,
            'idx_to_defect_type':  IDX_TO_DEFECT_TYPE,
            'idx_to_product_type': IDX_TO_PRODUCT_TYPE,
        }, V4_SAVE)
        if epoch >= EMA_START:
            ema.restore(model)
        print(f"\n  ✅ NEW BEST  {MONITOR}={best_score:.4f} → saved EdgeNet_V4_best.pth")
    else:
        no_improve += 1
        print(f"\n  (no improvement {no_improve}/{PATIENCE})")

    # ── Metrics JSON ────────────────────────────────────────────────────
    history.append({
        'epoch':          epoch,
        'train_f1':       tr['train_f1'],
        'train_def_acc':  tr['defect_acc'],
        'train_bin_acc':  tr['binary_acc'],
        'train_prod_acc': tr['product_acc'],
        'val_f1':         vl['val_f1'],
        'val_def_acc':    vl['defect_acc'],
        'val_bin_acc':    vl['binary_acc'],
        'val_prod_acc':   vl['product_acc'],
        'per_f':          per_f.tolist(),
        'per_p':          per_p.tolist(),
        'per_r':          per_r.tolist(),
        'best_score':     best_score,
        'lr_backbone':    optimizer.param_groups[0]['lr'],
        'lr_ca':          optimizer.param_groups[1]['lr'],
        'lr_head':        optimizer.param_groups[2]['lr'],
    })
    with open(METRICS_JSON, 'w') as f:
        json.dump({'class_names': SEMANTIC_GROUPS,
                   'best_score':  best_score,
                   'history':     history}, f, indent=2)

    # ── Early stopping ───────────────────────────────────────────────────
    if no_improve >= PATIENCE:
        print(f"\n⏹  Early stopping at epoch {epoch}")
        break

# ════════════════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"EdgeNetV4 TRAINING COMPLETE")
print(f"  Best Val {MONITOR} : {best_score:.4f}")
print(f"  Epochs run         : {len(history)}")
print(f"  Time               : {(time.time() - t0) / 60:.1f} min")
print(f"  Saved model        : {V4_SAVE}")
print(f"  Saved metrics JSON : {METRICS_JSON}")
print(f"{'='*60}")
print("✅ CELL 5 COMPLETE")

  Computing inverse-frequency class weights from training data...


NameError: name 'train_3head_loader' is not defined

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CELL 9 — RESULTS TABLE + STAIRCASE PLOT + PER-CLASS F1 HEATMAP
# Run this after Cell 8 finishes (or partially — shows whatever is done).
# ════════════════════════════════════════════════════════════════════════

# ── Load all saved results ──────────────────────────────────────────────
loaded = []
for cfg in ABLATION_CONFIGS:
    p = RESULTS_DIR / f'{cfg["name"]}.json'
    if p.exists():
        loaded.append(json.loads(p.read_text()))

if not loaded:
    print('No results yet. Run Cell 8 first.')
else:
    df_r = pd.DataFrame(loaded)
    base_f1 = df_r.iloc[0]['macro_f1']
    df_r['delta_f1'] = df_r['macro_f1'] - base_f1

    # ── Metrics table ──────────────────────────────────────────────────
    W = 95
    print('\n' + '='*W)
    print(f'{"Config":<35} {"Macro F1":>9} {"ΔF1":>8} {"DefAcc%":>8} '
          f'{"BinAcc%":>8} {"ProdAcc%":>9} {"Params M":>9} {"ms":>6} {"FPS":>6}')
    print('-'*W)
    for _, row in df_r.iterrows():
        print(f'{row["name"]:<35} '
              f'{row["macro_f1"]:>9.4f} {row["delta_f1"]:>+8.4f} '
              f'{row["defect_acc"]:>8.2f} {row["binary_acc"]:>8.2f} '
              f'{row["product_acc"]:>9.2f} {row["params_M"]:>9.2f} '
              f'{row["latency_ms"]:>6.1f} {row["fps"]:>6.0f}')
    print('='*W)
    print(f'\nBest: {df_r.loc[df_r.macro_f1.idxmax(), "name"]}  '
          f'F1={df_r.macro_f1.max():.4f}')

    # ── Staircase bar chart ─────────────────────────────────────────────
    names = [r['name'].replace('_', '\n') for r in loaded]
    f1s   = [r['macro_f1'] for r in loaded]

    fig, ax = plt.subplots(figsize=(14, 5))
    bars = ax.bar(range(len(f1s)), f1s,
                  color=['#2196F3' if i < len(f1s)-1 else '#4CAF50' for i in range(len(f1s))],
                  alpha=0.85, edgecolor='black', linewidth=0.7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=8)
    ax.set_ylabel('Macro F1 (test set)', fontsize=11)
    ax.set_title('EdgeNetV4-B2 Ablation Study — Cumulative Component Build-Up\n(50 epochs each · seed 42 · same 70/20/10 split)', fontsize=11)
    lo = max(0, min(f1s) - 0.05)
    ax.set_ylim(lo, min(1.0, max(f1s) + 0.06))
    for i, (bar, v) in enumerate(zip(bars, f1s)):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
        if i > 0:
            delta = v - f1s[i-1]
            color = 'green' if delta >= 0 else 'red'
            ax.text(bar.get_x() + bar.get_width()/2, lo + 0.005,
                    f'{delta:+.3f}', ha='center', va='bottom', fontsize=7, color=color)
    ax.axhline(y=f1s[0], color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='Baseline')
    ax.legend(fontsize=9)
    plt.tight_layout()
    out_fig = OUTPUT_DIR / 'ablation_staircase.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_fig}')

    # ── Per-class F1 heatmap (only when all 7 are done) ──────────────────
    if len(loaded) == 7:
        mat = np.array([r['per_class_f1'] for r in loaded])
        fig2, ax2 = plt.subplots(figsize=(13, 5))
        im = ax2.imshow(mat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
        ax2.set_xticks(range(NUM_DEFECT_TYPES))
        ax2.set_xticklabels(SEMANTIC_GROUPS, rotation=30, ha='right', fontsize=9)
        ax2.set_yticks(range(7))
        ax2.set_yticklabels([r['name'] for r in loaded], fontsize=8)
        plt.colorbar(im, ax=ax2, label='F1 score', shrink=0.8)
        ax2.set_title('Per-Class F1 Matrix — All 7 Ablation Configs', fontsize=11)
        for i in range(7):
            for j in range(NUM_DEFECT_TYPES):
                ax2.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center',
                         fontsize=7, color='black')
        plt.tight_layout()
        out_hm = OUTPUT_DIR / 'ablation_per_class_f1.png'
        plt.savefig(out_hm, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved → {out_hm}')
    else:
        print(f'({len(loaded)}/7 runs done — per-class heatmap will appear when all 7 complete)')